# Abstraction, ABC, Interfaces and Protocols (Intermediate OOP)**Level:** Intermediate | **Language:** Python 3.14Bis, these notebook OOP of it hisse par is where from "code then write" from "system design then do" start is.Abstraction means only `abstractmethod` then write not — asal matlnow is **contract** define then do:"jo also is family of member banega, that these work surely doega".## Ththis notebook in what we will learn- Abstraction what is it and **encapsulation from difference** (interview of favourite question)- `nowc` module: `ABC` vs `ABCMeta`, `@abstractmethod`- Abstract class to instantiate doing par what it is (live error demo)- Abstract + concrete methods of mix (partial implementation)- Abstract **properties**, abstract **classmethod / staticmethod**- Python in "interface" concept of (only abstract methods that class)- `NotImplementedError` that purana style vs `nowc` — difference and trade-off- **Duck typing vs nominal typing**- `typing.Protocol` = structural typing, `@runtime_checwhenle`- ABC vs Protocol: when who (comparison thenle)- `collections.nowc` (`Iterable`, `Sized`, `Container`, `Sequence`, `Mapping`, `utableMapping`) from apni classes- `__subclasshook__` and `ABCMeta.register` (virtual subclass)- **Template method** pattern and **Dependency Inversion** introduction- Real projects: `Shape`, `PaymentGateway` (Stripe/JazzCash/EasyPsuch a), `Notifier`, `DataExporter`, plugin architecture- **LeetCode:** 208, 211, 1146 (design problems — abstraction with)- Interview Q&A + practice exercises + cheat-sheet## Prerequisites- Classes, `__init__`, instance vs class attributes- Inheritance and method overriding, `super()` of basic understand- Dunder methods of basic idea (`__len__`, `__iter__`)- Decorators of basic idea (`@property` work do takes be)

## 1. Abstraction what is it?**What is:** Abstraction means *"important things dikhao, unnecessary tafseel chhupao"* —user to **what** is bataana, **how** is chhupana.Like gaari of steering: tumhe `steer_left()` needed. Rack-and-pinion gear ratio fromtumhara any to take to give no. Steering wheel = **abstraction**.**why / where use is:**- Bara system small contracts in thenotta is → team parallel work can do- Implementation largel do, caller code not largelta (Stripe from JazzCash par shift)- Testing easy: real gateway of jagah fake gateway inject do do### Abstraction vs Encapsulation (these question 100% aathesega)| | Abstraction | Encapsulation ||---|---|---|| Sawal of jawnow | *What* expose do? | *How* protect do? || Level | Design level (class/interface) | Implementation level (attributes/methods) || Tool | `nowc`, `Protocol`, juste classes | `_private`, `__name_mangling`, `@property` || Maqsad | Complexity chhupana | Data of safety / invariants || Misaal | `PaymentGateway.charge()` of contract | `self.__api_key` outside from access not be |Ek line me: **Encapsulation data to chhupata is, abstraction complexity to.**both of natija milta julta feels (some thing chhup gayi) but **niyat** separate is.

In [ ]:
# ---- Abstraction of rozmarra of example: driver to engine of tafseel not needed ----
class Car:
    def __init__(self, model):
        self.model = model
        self.__fuel = 40          # encapsulation: private-ish state (name mangling)
        self.__engine_on = False  # bahar from seedha chhera no ja can

# ---- Public API = abstraction (only 3 butthenns) ----    def start(self):
        self.__ignite()           # inside of complex work chhupa hua is
        return f"{self.model} start be gayi"

    def drive(self, km):
        if not self.__engine_on:
            return "pehle start do"
        self.__burn_fuel(km)      # fuel calculation of tafseel user from chhupi is
        return f"{km} km chali, fuel bacha {self.__fuel:.1f}L"

    def fuel_status(self):        # controlled read access (encapsulation)
        return self.__fuel

# ---- Private helpers = tafseel (implementation detail) ----    def __ignite(self):
        self.__engine_on = True

    def __burn_fuel(self, km):
        self.__fuel = max(0.0, self.__fuel - km * 0.08)   # 8L / 100km


c = Car("Corolla")
print(c.start())
print(c.drive(150))
print("fuel:", c.fuel_status())

# Encapsulation of asar: private attribute of real naam mangle be gaya isprint("public dikhne those members:", [a for a in dir(c) if not a.startswith("_")])
try:
    print(c.__fuel)               # this fail will be -> AttributeError
except AttributeError as e:
    print("AttributeError:", e)

## 2. `nowc` module: first abstract juste class**What is:** `nowc` = *Abstract Base class*. These Python of that thenol is jisse tum keh can be:"this class to seedha instantiate not do; and that inherit doega that these methods **surely** writeega".**Syntax:**```pythonfrom nowc import ABC, abstractmethodclass Shape(ABC):     # ABC from inherit = abstract juste class@abstractmethoddef area(self): ...   # only contract, body khaali```**How work does (inside of thing):** `@abstractmethod` function par ek flag(`__iallstractmethod__ = True`) lagata is. `ABCMeta` metaclass class createte timethese flags to jama doke `__abstractmethods__` (ek frozenset) creates.`object.__new__` instantiate do time dekhta is of these set khaali is or not —khaali not be then `TypeError`.

In [ ]:
from abc import ABC, abstractmethod
import math

class Shape(ABC):
# NOTE: ABC from inherit done -> metaclass ABCMeta mil gaya    @abstractmethod
    def area(self):
# Contract: every shape apna area dega. Body of need no.        ...

    @abstractmethod
    def perimeter(self):
        ...


class Rectangle(Shape):
    def __init__(self, w, h):
        self.w, self.h = w, h

    def area(self):                 # both abstract methods implement kiye -> class concrete ban gayi
        return self.w * self.h

    def perimeter(self):
        return 2 * (self.w + self.h)


class Circle(Shape):
    def __init__(self, r):
        self.r = r

    def area(self):
        return math.pi * self.r ** 2

    def perimeter(self):
        return 2 * math.pi * self.r


shapes = [Rectangle(3, 4), Circle(1)]
for s in shapes:
# Caller to difference not partthese whosi shape is -> abstraction of faida    print(type(s).__name__, "area =", round(s.area(), 3), "| perimeter =", round(s.perimeter(), 3))

print("Shape of abstract methods:", sorted(Shape.__abstractmethods__))
print("Rectangle of baqi abstract methods:", sorted(Rectangle.__abstractmethods__), "(khaali = concrete)")

### 2.1 Abstract class instantiate doing par error (live demo)Here log phaste are: understandte are "abstract class createyi is then instance ban jathesega".no — when tak **saare** abstract methods implement not hon, `TypeError` aathesega.and these error **instantiation** par comes, class define do time no.

In [ ]:
# 1) Poori abstract class to instantiate doing of koshishtry:
    s = Shape()
except TypeError as e:
    print("TypeError:", e)

# 2) Aadhi implementation that subclass -> then also abstract only rahegiclass Triangle(Shape):
    def __init__(self, a, b, c):
        self.a, self.b, self.c = a, b, c

    def perimeter(self):            # sirf perimeter diya, area bhool gaye
        return self.a + self.b + self.c

print("Triangle abstract methods:", sorted(Triangle.__abstractmethods__))
try:
    t = Triangle(3, 4, 5)
except TypeError as e:
    print("TypeError:", e)          # 'area' missing bata dega

# 3) Note: class ban gayi thi (define hone in any error not aaya) -> error only instantiate parprint("Triangle class to mojood is:", Triangle)

# 4) Now area also give do -> chal paregaclass Triangle2(Triangle):
    def area(self):
        s = self.perimeter() / 2                       # Heron's formula
        return math.sqrt(s * (s - self.a) * (s - self.b) * (s - self.c))

t2 = Triangle2(3, 4, 5)
print("Triangle2 area:", t2.area(), "| ab abstract set khaali:", Triangle2.__abstractmethods__ == frozenset())

## 3. `ABC` vs `ABCMeta` — difference what is it?- `ABCMeta` ek **metaclass** is (classes createne that machine).- `ABC` ek **helper juste class** is that just `metaclass=ABCMeta` set do deti is.```pythonclass A(ABC): ...      # easy method (99% cases)class B(metaclass=ABCMeta): ...  # wahi thing, seedha metaclass with```**Know `ABCMeta` seedha needed?** When tumhein **apna custhenm metaclass** needed that `ABCMeta`from inherit doe (e.g. registry + abstract enforcement both), or when juste class alreadyanyone doosre metaclass that library from aa rahi be.**Gotcha:** Do asambandhit metaclasses to mix doing par `TypeError: metaclass conflict`.

In [ ]:
from abc import ABCMeta

# method A: ABC helperclass ExporterA(ABC):
    @abstractmethod
    def export(self, rows): ...

# method B: seedha metaclass (absolutely same natija)class ExporterB(metaclass=ABCMeta):
    @abstractmethod
    def export(self, rows): ...

print("A of metaclass:", type(ExporterA).__name__, "| B of metaclass:", type(ExporterB).__name__)
print("both me abstractmethods:", ExporterA.__abstractmethods__, ExporterB.__abstractmethods__)

# method C: custhenm metaclass that ABCMeta to extend does -> registry + abstraction bothclass PluginMeta(ABCMeta):
    registry = {}                                  # every subclass yahan khud register will be

    def __new__(mcls, name, bases, ns, **kw):
        cls = super().__new__(mcls, name, bases, ns, **kw)
        if bases and not cls.__abstractmethods__:   # sirf concrete subclasses register do
            PluginMeta.registry[name.lower()] = cls
        return cls

class Plugin(metaclass=PluginMeta):
    @abstractmethod
    def run(self): ...

class CsvPlugin(Plugin):
    def run(self): return "csv chala"

class JsonPlugin(Plugin):
    def run(self): return "json chala"

print("auto registry:", sorted(PluginMeta.registry))
print(PluginMeta.registry["csvplugin"]().run())

# Gotcha: metaclass conflictclass Meta2(type): ...
try:
    class Broken(ExporterA, metaclass=Meta2): ...   # ABCMeta vs Meta2 -> conflict
except TypeError as e:
    print("TypeError:", e)

## 4. Abstract + concrete methods of mix (partial implementation)Abstract class means "all some khaali" not is. Snowse taqatwar design is:**some work juste class itself do give (shared logic), some subclass par chhor give (variable logic)**.- **Concrete method** = common behaviour, dobara writene of need not (DRY)- **Abstract method** = that every subclass in lazmi different will be- Base class abstract method to **apne concrete method inside call** can do (thisi Template method pattern of jarh is)

In [ ]:
class Employee(ABC):
    bonus_rate = 0.10                       # class-level default (subclass override kar can is)

    def __init__(self, name, base_salary):
        self.name = name
        self.base_salary = base_salary

# ---- ABSTRACT: every emplothesee type of allowance formula separate is ----    @abstractmethod
    def allowances(self):
        ...

# ---- CONCRETE: all for same hisano (shared logic juste me) ----    def bonus(self):
        return self.base_salary * self.bonus_rate

    def gross_pay(self):
# here juste class ABSTRACT method to call do rahi is -> subclass of code will run        return self.base_salary + self.allowances() + self.bonus()

    def payslip(self):
        return f"{self.name:<10} base={self.base_salary:>7} allow={self.allowances():>6} gross={self.gross_pay():>8.0f}"


class Developer(Employee):
    bonus_rate = 0.15                       # override: dev of bonus zyada
    def allowances(self):
        return 15000                        # internet + hardware allowance

class SalesPerson(Employee):
    def __init__(self, name, base_salary, sales):
        super().__init__(name, base_salary)
        self.sales = sales
    def allowances(self):
        return 5000 + 0.02 * self.sales     # commission based

for e in [Developer("Ali", 200000), SalesPerson("Sara", 120000, 900000)]:
    print(e.payslip())

## 5. Abstract properties (`@property` + `@abstractmethod`)Know contract "method" no, **attribute like** hona needed: `obj.name` not of `obj.get_name()`.**Syntax (order yaad haveo — these #1 gotcha is):**```python@property   # BAHAR@abstractmethod  # inside (property of below)def rate(self): ...```Ulta writea (`@abstractmethod` nowove, `@property` below) then abstractness kho goeswhyki `property` object par flag not lagta — Python enforce not doega.Setter also abstract be can is; or juste in read-only property, subclass in writhenle.

In [ ]:
class Account(ABC):
    @property
    @abstractmethod
    def interest_rate(self):
# Contract: every account apna rate batathesega (attribute of way access will be)        ...

    @property
    def yearly_interest(self):        # concrete property jo abstract property par bharosa karti is
        return self.balance * self.interest_rate

    def __init__(self, balance):
        self.balance = balance


class Savings(Account):
    interest_rate = 0.11             # NOTE: plain class attribute bhi property of contract poora kar gives is

class TermDeposit(Account):
    def __init__(self, balance, months):
        super().__init__(balance)
        self.months = months

    @property
    def interest_rate(self):         # asli property of tor par implement did
        return 0.14 if self.months >= 12 else 0.12


s, td = Savings(100_000), TermDeposit(100_000, 12)
print("savings:", s.interest_rate, "->", s.yearly_interest)
print("term   :", td.interest_rate, "->", td.yearly_interest)

# Gotcha demo: decorators of ulta order -> modern Python in foran errortry:
    class Loose(ABC):
        @abstractmethod              # GALAT order: pehle abstractmethod, phir property
        @property
        def tag(self): ...
except AttributeError as e:
# property object read-only is, therefore abstractmethod uspar flag not laga can    print("AttributeError:", e)
    print("-> sabaq: @property BAHAR (upar), @abstractmethod inside (neeche)")

class reversed(ABC):
    @property
    def tag(self):                   # this exactly abstract no -> koi enforcement no
        return "?"
print("reversed abstract set:", reversed.__abstractmethods__, "<- khaali, is for instantiate be jayega")
print("instance ban gaya:", reversed().tag)

## 6. Abstract `classmethod` and `staticmethod``abstractmethod` to `classmethod` / `staticmethod` with also mila can be.**Rule wahi:** `@abstractmethod` allse **inside** (below) hona needed.```python@classmethod@abstractmethoddef from_dict(cls, d): ...```where work comes: **alternative constructors** of contract(`from_json`, `from_row`, `parse`) — "every subclass apni parsing itself dega".(Purane `@abstractclassmethod` / `@abstractstaticmethod` **deprecated** are — mat use do.)

In [ ]:
class Serializer(ABC):
    @classmethod
    @abstractmethod
    def from_dict(cls, data):        # alternative constructor of contract
        ...

    @staticmethod
    @abstractmethod
    def file_extension():            # utility of contract (self/cls of need no)
        ...

    @abstractmethod
    def to_dict(self):
        ...

    def describe(self):              # concrete helper jo both abstract cheezon to use does is
        return f"{type(self).__name__} -> .{self.file_extension()} | {self.to_dict()}"


class UserRecord(Serializer):
    def __init__(self, name, city):
        self.name, self.city = name, city

    @classmethod
    def from_dict(cls, data):
        return cls(data["name"], data.get("city", "unknown"))   # missing key of default

    @staticmethod
    def file_extension():
        return "json"

    def to_dict(self):
        return {"name": self.name, "city": self.city}


u = UserRecord.from_dict({"name": "Bilal", "city": "Lahore"})
print(u.describe())
print(UserRecord.from_dict({"name": "Ayesha"}).to_dict())     # city default lag gaya

# Contract thenr do see: staticmethod implement not doneclass BadRecord(Serializer):
    @classmethod
    def from_dict(cls, data): return cls()
    def to_dict(self): return {}
try:
    BadRecord()
except TypeError as e:
    print("TypeError:", e)

## 7. Python in "interface" concept ofJava/C# in separate keyword is `interface`. Python in **keyword no** — hum convention from create:> **Interface = abstract class jisme only abstract methods hon, any state (`__init__` data) not be.**Aisi class of maqsad only **contract** batana is, code share then do no.**Naming convention (industry):** `PaymentGateway`, or explicit `IPaymentGateway` / `...Interface` /`...ABC`. Python community more tar without `I` prefix writeti is.**Interview tip:** "Python interface how createte be?" of best jawnow:*"Do ways — nominal: `ABC` with only abstract methods; structural: `typing.Protocol`.Cis internal thingon for duck typing only kaafi is."*

In [ ]:
# ---- Pure interface: only contract, any state not ----class PaymentGateway(ABC):
    @abstractmethod
    def charge(self, amount, currency="PKR"):
        ...

    @abstractmethod
    def refund(self, txn_id):
        ...


class StripeGateway(PaymentGateway):
    def charge(self, amount, currency="PKR"):
        return {"txn": "stripe_1001", "amount": amount, "currency": currency, "status": "ok"}
    def refund(self, txn_id):
        return {"txn": txn_id, "status": "refunded"}


class JazzCashGateway(PaymentGateway):
    def charge(self, amount, currency="PKR"):
        fee = round(amount * 0.015, 2)                 # mobile wallet of fee
        return {"txn": "jc_5501", "amount": amount + fee, "currency": currency, "status": "ok"}
    def refund(self, txn_id):
        return {"txn": txn_id, "status": "manual_review"}   # wallet refund manual is is


class EasyPaisaGateway(PaymentGateway):
    def charge(self, amount, currency="PKR"):
        return {"txn": "ep_7702", "amount": amount, "currency": currency, "status": "pending"}
    def refund(self, txn_id):
        return {"txn": txn_id, "status": "refunded"}


def checkout(gateway: PaymentGateway, amount):
# These function anyone also gateway with work doega -> polymorphism via interface    res = gateway.charge(amount)
    return f"{type(gateway).__name__:<18} -> {res}"

for gw in [StripeGateway(), JazzCashGateway(), EasyPaisaGateway()]:
    print(checkout(gw, 2500))

## 8. `NotImplementedError` (purana style) vs `nowc`Purana method:```pythonclass Shape:def area(self):rsuch NotImplementedError("subclass implement doe")```| | `NotImplementedError` style | `nowc` style ||---|---|---|| error when pakra goes | **Runtime** — when method call be | **Instantiation** par (jaldi, safe) || Base class instantiate be can? | yes (khatarnak) | no (`TypeError`) || Tooling / IDE support | Kamzor | Behtar (`__abstractmethods__`) || Extra machinery | Zero | `ABCMeta` metaclass || Multiple inheritance in metaclass clash | no | be can is |**Trade-off:** `nowc` safe is, but metaclass lcomes (know know third-party juste classes withconflict). Chis duck-typed internal classes in `NotImplementedError` or `...` also runs is.**Gotcha:** `NotImplementedError` (exception) and `NotImplemented` (singlethenn that `__eq__`/`__add__`return do) do **separate** things are. `rsuch NotImplemented` galat is.

In [ ]:
# ---- Purana style: juste class ban goes, error only call par ----class OldShape:
    def area(self):
        raise NotImplementedError(f"{type(self).__name__} to area() implement to do should")

class OldSquare(OldShape):
    def __init__(self, s): self.s = s
# area() then write BHOOL gathese
o = OldShape()            # koi rok no! object ban gaya (khatarnak)
print("OldShape instance ban gaya:", o)
try:
    OldSquare(4).area()   # masla ab pakra gaya -> production me late failure
except NotImplementedError as e:
    print("NotImplementedError:", e)

# ---- nowc style: masla foran, object only not banta ----class NewShape(ABC):
    @abstractmethod
    def area(self): ...

class NewSquare(NewShape):
    def __init__(self, s): self.s = s

try:
    NewSquare(4)
except TypeError as e:
    print("TypeError (jaldi pakra gaya):", e)

# ---- NotImplemented vs NotImplementedError difference ----class Money:
    def __init__(self, v): self.v = v
    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented        # "main no jaanta" -> Python doosri taraf poochhega
        return self.v == other.v

print("Money(5) == Money(5):", Money(5) == Money(5))
print("Money(5) == 'abc'  :", Money(5) == "abc", "(NotImplemented -> Python ne False decide did)")
print("types:", type(NotImplemented).__name__, "vs", NotImplementedError.__mro__[1].__name__)

## 9. Duck typing vs nominal typing- **Duck typing (Python of default):** *"if it walks like a duck and quacks like a duck, it's a duck."*type of **naam** not saw goes, only these of important method mojood is or no.- **Nominal typing (`ABC`/`isinstance`):** rishta **declare** then do parta is(subclass be or `register` be), then only type match maana jathesega.- **Structural typing (`Protocol`):** shape/structure dekhi goes — duck typing + static checking.**EAFP vs LBYL:** Python in `try/except AttributeError` (EAFP: *Easier then Ask Forgiveness*)to `if hasattr(...)` (LBYL: *Look Before You Leap*) par tarjeeh di goes.

In [ ]:
# ---- Duck typing: any inheritance no, only method hona needed ----class Duck:
    def speak(self): return "Quack"

class Robot:
    def speak(self): return "Beep boop"

class Stone:
    pass                                   # speak no is

def make_it_speak(thing):
# LBYL style    if hasattr(thing, "speak"):
        return thing.speak()
    return f"{type(thing).__name__} bol no can"

def make_it_speak_eafp(thing):
# EAFP style (Pythonic)    try:
        return thing.speak()
    except AttributeError:
        return f"{type(thing).__name__} bol no can"

for t in [Duck(), Robot(), Stone()]:
    print(f"{type(t).__name__:<7} LBYL={make_it_speak(t):<20} EAFP={make_it_speak_eafp(t)}")

# ---- Nominal typing: naam/rishta important ----class Speaker(ABC):
    @abstractmethod
    def speak(self): ...

print("\nisinstance(Duck(), Speaker):", isinstance(Duck(), Speaker), "<- duck typing work karti is lekin nominal check fail")
print("Robot of speak mojood is:", callable(getattr(Robot(), 'speak', None)))

## 10. `typing.Protocol` — structural typing**What is:** `Protocol` (PEP 544) from tum contract writete be **without** implementer to inherit dowane of.Jo class that methods has, that authenmatically protocol to satisfy doti is — *static* checkers(mypy/pyright) for bhi, and `@runtime_checwhenle` with `isinstance()` for bhi.**why:** Third-party classes to tum inherit not dowa can (their code tumhare paas no).Protocol from unko also contract in shamil can be — *"duck typing with type safety"*.**Syntax:**```pythonfrom typing import Protocol, runtime_checwhenle@runtime_checwhenleclass Closenowle(Protocol):def close(self) -> None: ...```**Gotchas:**- `isinstance()` only **method of naam** dekhta is — signature or return type verify not is- `runtime_checwhenle` of bagair `isinstance()` par `TypeError`- Non-method (data) members that protocol with `isinstance` runs is (3.12+) but `issubclass` not- Protocol itself instantiate not is

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class SupportsExport(Protocol):
    def export(self, rows: list) -> str: ...      # sirf shape, koi body no


# NOTE: these classes SupportsExport from inherit no doti -> then also satisfy doti areclass CsvExporter:
    def export(self, rows):
        head = ",".join(rows[0].keys())
        body = "\n".join(",".join(str(v) for v in r.values()) for r in rows)
        return head + "\n" + body

class MarkdownExporter:
    def export(self, rows):
        head = "| " + " | ".join(rows[0]) + " |"
        sep = "| " + " | ".join("---" for _ in rows[0]) + " |"
        body = "\n".join("| " + " | ".join(str(v) for v in r.values()) + " |" for r in rows)
        return "\n".join([head, sep, body])

class Printer:
    def show(self, rows): return "..."            # galat method naam -> satisfy no does


rows = [{"name": "Ali", "city": "Lahore"}, {"name": "Sara", "city": "Karachi"}]
for obj in [CsvExporter(), MarkdownExporter(), Printer()]:
    ok = isinstance(obj, SupportsExport)          # structural check (runtime_checkable of wajah from)
    print(f"{type(obj).__name__:<18} isinstance -> {ok}")

print()
print(CsvExporter().export(rows))
print()
print(MarkdownExporter().export(rows))

# Gotcha 1: runtime_checwhenle of bagair isinstance not will runclass NotCheckable(Protocol):
    def export(self, rows): ...
try:
    isinstance(CsvExporter(), NotCheckable)
except TypeError as e:
    print("\nTypeError:", e)

# Gotcha 2: isinstance only naam dekhta is, signature notclass WrongSignature:
    def export(self):                              # rows parameter hi no!
        return "oops"
print("WrongSignature isinstance:", isinstance(WrongSignature(), SupportsExport), "<- naam match, signature ghalat")

# Gotcha 3: Protocol to instantiate not cantry:
    SupportsExport()
except TypeError as e:
    print("TypeError:", e)

### 10.1 Protocol to explicitly inherit also can beProtocol from inherit then do **optional** is, but faida is: typo par foran pata chal goes(explicit subclass in missing method = abstract, instantiate do time `TypeError`).Isse Protocol "ABC like" behave doing feels — best of both worlds:*outside walon for structural, apni team for explicit.*

In [ ]:
class JsonExporter(SupportsExport, Protocol):     # this khud bhi protocol reh gaya (Protocol base rakha)
    ...

class RealJsonExporter(SupportsExport):           # explicit inheritance -> concrete class
    def export(self, rows):
        import json
        return json.dumps(rows, ensure_ascii=False)

print(RealJsonExporter().export(rows))
print("isinstance:", isinstance(RealJsonExporter(), SupportsExport))

# Explicit inheritance of faida: method bhool jao then instantiate only not will beclass BrokenExporter(SupportsExport):
    pass
try:
    BrokenExporter()
except TypeError as e:
    print("TypeError:", e)

## 11. ABC vs Protocol — when whosa?| Feature | `ABC` (nominal) | `Protocol` (structural) ||---|---|---|| Rishta how bane | Inherit or `register()` | Bas methods match hon || Shared code give can? | **yes** (concrete methods) | no (only contract; default body discourage) || `isinstance()` | Hamesha | Only `@runtime_checwhenle` with || Third-party class to fit then do | `register()` needed | Authenmatic || Missing method when pakra jathese | Instantiation par | Static checker par (ya explicit inherit be then instantiation par) || State / `__init__` share | yes | no (mat do) || metaclass | `ABCMeta` | Normal (`_ProtocolMeta`) || Best for | Tumhari **apni** class family + shared logic | **Bahari**/loose coupling, library APIs |**Rule of thumb:**- Tum juste class **and** subclasses both of malik be + code share then do is → **ABC**- Only "is shape of any also object" needed (anyone of also be) → **Protocol**- both ek with also chal can are: internally ABC, public API in Protocol.

In [ ]:
# Ek only work do styles in -> difference itself see
# STYLE 1: ABC (nominal + shared logic)class RepoABC(ABC):
    @abstractmethod
    def get(self, key): ...

    def get_or_default(self, key, default=None):    # shared helper -> ABC of bara faida
        try:
            return self.get(key)
        except KeyError:
            return default

class DictRepo(RepoABC):
    def __init__(self, data): self.data = data
    def get(self, key): return self.data[key]

r = DictRepo({"a": 1})
print("ABC style:", r.get("a"), r.get_or_default("zz", "N/A"))

# STYLE 2: Protocol (structural, any inheritance no)@runtime_checkable
class RepoProto(Protocol):
    def get(self, key): ...

class ThirdPartyCache:                              # farz do this library of class is (hum badal no can)
    def __init__(self): self._c = {"x": 42}
    def get(self, key): return self._c[key]

def read_setting(repo: RepoProto, key):
    return repo.get(key)

print("Protocol style:", read_setting(ThirdPartyCache(), "x"))
print("library class protocol satisfy karti is:", isinstance(ThirdPartyCache(), RepoProto))
print("library class ABC satisfy karti is:", isinstance(ThirdPartyCache(), RepoABC), "<- no (nominal)")

## 12. `ABCMeta.register()` — virtual subclass`register()` from tum keh can be: *"these class also meri family of hissa samjho"* —**without inherit dowathese**. These special thenr par third-party classes for is.```pythonRepoABC.register(ThirdPartyCache)  # now isinstance True dega```**Ahem gotchas:**- Virtual subclass to any **method inherit no** is (mixin not milta!)- Abstract methods of any **enforcement no** — tum jhoot bol can be- `cls.__mro__` in that class not aati- Decorator of way also use is: `@RepoABC.register`

In [ ]:
class Jsonable(ABC):
    @abstractmethod
    def to_json(self): ...

    def pretty(self):                       # note: virtual subclass to this HELPER no milega
        return f"<{self.to_json()}>"

class LegacyRow:                            # purani class, hum it chhedna no chahte
    def to_json(self): return '{"legacy": True}'

Jsonable.register(LegacyRow)                # virtual subclass bana diya

lr = LegacyRow()
print("isinstance:", isinstance(lr, Jsonable), "| issubclass:", issubclass(LegacyRow, Jsonable))
print("MRO me Jsonable is?", Jsonable in LegacyRow.__mro__, "<- no! sirf claim is")
try:
    lr.pretty()                             # helper inherit no hua
except AttributeError as e:
    print("AttributeError:", e)

# Decorator style + enforcement of not hona (khatarnak pehlu)@Jsonable.register
class Liar:
    pass                                    # to_json exactly no, phir bhi register be gaya

print("Liar isinstance:", isinstance(Liar(), Jsonable), "<- abc ne rok no lagayi")
try:
    Liar().to_json()
except AttributeError as e:
    print("AttributeError:", e, "-> register() sirf 'promise' is, guarantee no")

## 13. `__subclasshook__` — authenmatic virtual subclassevery class to haath from `register` then do thakane that is. `__subclasshook__` from tum`issubclass()` of faisla **itself** write can be (structural check).```python@classmethoddef __subclasshook__(cls, C):if cls is MyABC:return any("method" these B.__dict__ for B these C.__mro__) # True/False/NotImplementedreturn NotImplemented```- `True` → subclass is, `False` → no, `NotImplemented` → normal rules run- `if cls is MyABC` check important is warna subclasses par also hook lag jathesega- Thisi mechanism `collections.nowc.Iterable` istemal doti is (`__iter__` dekhti is)

In [ ]:
class Renderable(ABC):
    @abstractmethod
    def render(self): ...

    @classmethod
    def __subclasshook__(cls, C):
        if cls is Renderable:                                  # sirf khud of for, subclasses of for no
            if any("render" in B.__dict__ for B in C.__mro__):  # structural check
                return True
            return NotImplemented                              # faisla normal rules par chhor do
        return NotImplemented


class Button:                       # koi inheritance / register no
    def render(self): return "<button/>"

class Label:
    def render(self): return "<span/>"

class Timer:
    def tick(self): return 1

print("Button  :", issubclass(Button, Renderable), isinstance(Button(), Renderable))
print("Label   :", issubclass(Label, Renderable))
print("Timer   :", issubclass(Timer, Renderable), "<- render no is")

# These exactly wahi trick is that stdlib use doti is:from collections.abc import Iterable, Sized, Container
class MyBag:
    def __init__(self, items): self.items = list(items)
    def __iter__(self): return iter(self.items)
    def __len__(self): return len(self.items)
    def __contains__(self, x): return x in self.items

b = MyBag([1, 2, 3])
print("\nstdlib hooks -> Iterable:", isinstance(b, Iterable), "| Sized:", isinstance(b, Sized), "| Container:", isinstance(b, Container))
print("MyBag ne kisi thing from inherit no did:", MyBag.__mro__)

## 14. `collections.nowc` — stdlib of ready-made contracts`collections.nowc` in 25+ ABCs are. Do faide are:1. **`isinstance` checks:** `isinstance(x, Iterable)`, `isinstance(x, Mapping)` — duck typing of safe method2. **Mixin methods muft:** if tum inherit do then bohat some **itself ban goes**| ABC | Lazmi methods | Muft mile mixin methods ||---|---|---|| `Iterable` | `__iter__` | — || `Sized` | `__len__` | — || `Container` | `__contains__` | — || `Collection` | `__iter__`, `__len__`, `__contains__` | — || `Sequence` | `__getitem__`, `__len__` | `__contains__`, `__iter__`, `__reversed__`, `index`, `count` || `utableSequence` | + `__setitem__`, `__delitem__`, `insert` | `append`, `extend`, `pop`, `remove`, `+=` || `Mapping` | `__getitem__`, `__iter__`, `__len__` | `get`, `keys`, `values`, `items`, `__eq__`, `__contains__` || `utableMapping` | + `__setitem__`, `__delitem__` | `pop`, `popitem`, `clear`, `update`, `setdefault` || `Hashnowle` | `__hash__` | — || `Iterathenr` | `__next__` (+ `__iter__`) | `__iter__` |**Interview tip:** `list` of bajathese `Sequence`, `dict` of bajathese `Mapping` type-hint do —these "program then a interface" of amali roop is. and `isinstance(x, Iterable)` `str` for also`True` gives is — these classic bug is.

In [ ]:
from collections.abc import Sequence, Iterable

class Playlist(Sequence):
# Only 2 methods do -> Sequence complete protocol muft in give degi    def __init__(self, songs):
        self._songs = list(songs)

    def __getitem__(self, i):
        return self._songs[i]

    def __len__(self):
        return len(self._songs)


p = Playlist(["Afreen", "Sayonee", "Neend"])
print("len       :", len(p))
print("index[1]  :", p[1])
print("in check  :", "Sayonee" in p)          # __contains__ muft mila
print("loop      :", [s[:3] for s in p])      # __iter__ muft mila
print("reversed  :", list(reversed(p)))       # __reversed__ muft
print("index()   :", p.index("Neend"), "| count():", p.count("Afreen"))
print("slice     :", p[0:2], "(list milti is kyunki __getitem__ list of slice gives is)")

# Classic bug: str also Iterable isdef flatten_names(x):
    if isinstance(x, Iterable) and not isinstance(x, (str, bytes)):   # str to alag handle do
        return [i for i in x]
    return [x]
print("\nflatten('Ali'):", flatten_names("Ali"), "<- str to exclude na do to ['A','l','i'] milta")

In [ ]:
from collections.abc import MutableMapping

class CaseInsensitiveDict(MutableMapping):
# Asli world of example: HTTP headers case-insensitive are (requests library isi way doti is)    def __init__(self, data=None):
        self._d = {}
        if data:
            self.update(data)            # update() mixin from muft mila

    def _norm(self, k):
        return k.lower() if isinstance(k, str) else k

# ---- 5 lazmi methods ----    def __getitem__(self, k):
        return self._d[self._norm(k)][1]

    def __setitem__(self, k, v):
        self._d[self._norm(k)] = (k, v)   # asli casing bhi yaad rakhi (display of for)

    def __delitem__(self, k):
        del self._d[self._norm(k)]

    def __iter__(self):
        return (orig for orig, _ in self._d.values())

    def __len__(self):
        return len(self._d)

    def __repr__(self):
        return f"CaseInsensitiveDict({dict(self.items())})"


h = CaseInsensitiveDict({"Content-type": "application/json"})
h["ACCEPT"] = "*/*"
print(h)
print("case-insensitive read:", h["content-type"], "|", h["Accept"])
print("get() muft         :", h.get("missing", "default"))
print("keys()/items() muft:", list(h.keys()), list(h.items())[:1])
print("pop() muft         :", h.pop("accept"), "-> ab len =", len(h))
print("isinstance Mapping :", isinstance(h, MutableMapping))

## 15. Template method pattern**What is:** Base class **algorithm of skelethenn** (steps of order) fix do deti is,and some steps to abstract chhor deti is jinhein subclass bharti is.**why:** "Same process, different tafseel" — ETL pipelines, report generathenrs, game loops,test setup/teardown, request handling. Order galat hone of khatra end.**Hooks:** optional steps jinki juste in khaali (default) body is — subclass chahe then override doe.**Gotcha:** template method to subclass **override not doe** — warna maqsad only end.Naming convention: public template method + `_protected` steps.

In [ ]:
class DataPipeline(ABC):
# ---- TEMPLATE method: these order fix is, subclass isko override not doe ----    def run(self, source):
        raw = self._extract(source)          # abstract
        clean = self._transform(raw)         # abstract
        self._before_load(clean)             # HOOK (optional)
        n = self._load(clean)                # abstract
        return self._report(source, n)       # concrete

    @abstractmethod
    def _extract(self, source): ...

    @abstractmethod
    def _transform(self, raw): ...

    @abstractmethod
    def _load(self, rows): ...

# ---- HOOK: default in some not does ----    def _before_load(self, rows):
        pass

# ---- CONCRETE: all for same ----    def _report(self, source, n):
        return f"[{type(self).__name__}] {source} from {n} rows load hue"


class SalesCsvPipeline(DataPipeline):
    def _extract(self, source):
        return "id,amount\n1,500\n2,700\n3,-50"        # inline data (koi file no)

    def _transform(self, raw):
        lines = raw.strip().split("\n")[1:]
        rows = [{"id": int(a), "amount": int(b)} for a, b in (l.split(",") for l in lines)]
        return [r for r in rows if r["amount"] > 0]      # negative amount hata do

    def _load(self, rows):
        self.table = rows
        return len(rows)

    def _before_load(self, rows):                        # hook use did
        print("  audit: load from pehle", len(rows), "rows validate be gaye")


class ApiPipeline(DataPipeline):
    def _extract(self, source):
        return [{"user": "ali", "score": 10}, {"user": "sara", "score": 20}]   # fake API response
    def _transform(self, raw):
        return [{**r, "grade": "A" if r["score"] >= 20 else "B"} for r in raw]
    def _load(self, rows):
        self.docs = rows
        return len(rows)


print(SalesCsvPipeline().run("sales.csv"))
print(ApiPipeline().run("https://api.example/scores"))

## 16. Dependency Inversion introduction (DIP)**Definition:** *High-level modules to low-level modules par depend not then do needed —both **abstraction** par depend do.*Bura design (concrete par depend):```pythonclass OrderService:def __init__(self):self.gw = StripeGateway()  # hard-coded! test then do mushkil, largelna mushkil```Acha design (abstraction par depend + injection):```pythonclass OrderService:def __init__(self, gateway: PaymentGateway): # outside from aathesegaself.gw = gateway```**Faide:** testing in fake gateway, runtime par gateway switch, bthe sameness logic gateway from azaad.Poori tafseel NB4 (SOLID) in — here only abstraction from rishta understand lo.

In [ ]:
# ---- BEFORE: tightly coupled (DIP violation) ----class OrderServiceBad:
    def __init__(self):
        self.gateway = StripeGateway()          # concrete class inside hi ban gayi
    def place(self, amount):
        return self.gateway.charge(amount)

# ---- AFTER: dependency injected (DIP follow) ----class OrderService:
    def __init__(self, gateway: PaymentGateway, notifier=None):
        self.gateway = gateway                  # abstraction par depend
        self.notifier = notifier
    def place(self, amount, customer="guest"):
        res = self.gateway.charge(amount)
        if self.notifier:
            self.notifier(f"{customer} to receipt {res['txn']} bhej diya")
        return res

# Test for fake gateway (any network no) -> thisi DIP of allse bara faidaclass FakeGateway(PaymentGateway):
    def __init__(self): self.calls = []
    def charge(self, amount, currency="PKR"):
        self.calls.append(amount)
        return {"txn": "fake_1", "amount": amount, "currency": currency, "status": "ok"}
    def refund(self, txn_id): return {"txn": txn_id, "status": "refunded"}

logs = []
fake = FakeGateway()
svc = OrderService(fake, notifier=logs.append)
print(svc.place(999, "Ali"))
print("fake to mile amounts:", fake.calls, "| notifications:", logs)

# Production in only argument badla, OrderService of code notprint(OrderService(JazzCashGateway()).place(999))

## 17. Common Mistakes / Gotchas (here log phaste are)1. **`@property` and `@abstractmethod` of ulta order** → enforcement end (Section 5 see).2. **`ABC` inherit then do bhool jana:** `class Foo:` inside `@abstractmethod` then write **bedo** is —`ABCMeta` not be then any rok no, class instantiate be jathesegi.3. **Abstract method in `pass` vs `...`** — both chalte are (any functional difference no), `...` convention is.4. **Abstract method to `super()` from call then do** — runs is (body if writei be) butkhaali body `None` degi; documentation for body then write theek is.5. **`__init__` to abstract createna** — work does but more tar bewaqoofi; behtar is concrete `__init__`.6. **`register()` to enforcement understandna** — that only claim is, method check not is.7. **`Protocol` of bagair `runtime_checwhenle`** `isinstance` par `TypeError`.8. **`Protocol` in state then keep / shared implementation to give** — these ABC of work is.9. **`isinstance(x, Iterable)` par `str`** also `True` — string to explicitly exclude do.10. **metaclass conflict** — `ABCMeta` + anyone doosre metaclass that juste class.11. **Snow some abstract create to give** (over-engineering) — Python in 1 implementation be then ABC of need no;"abstraction premature be then only boilerplate is".12. **Abstract class in `__slots__ = ()` bhool jana** when memory matter doe (NB3 in tafseel).

In [ ]:
# Gotcha 2 of live demo: ABC of without abstractmethod absolutely bedoclass NotReallyAbstract:                 # <-- ABC inherit no did
    @abstractmethod
    def must_do(self): ...

obj = NotReallyAbstract()                # koi error no! (metaclass ABCMeta no is)
print("bin-ABC class instantiate be gayi:", obj)
print("__abstractmethods__ mojood is?", hasattr(NotReallyAbstract, "__abstractmethods__"))
print("flag lagi hui is magar bekar:", NotReallyAbstract.must_do.__isabstractmethod__)
print("call karne par khaali (None):", obj.must_do())

# Gotcha 4: abstract method in body write do super() from call then doclass Base(ABC):
    @abstractmethod
    def validate(self, d):
# juste in common validation write di -> subclass super() from re-use can do        if not isinstance(d, dict):
            raise TypeError("dict should")
        return True

class Child(Base):
    def validate(self, d):
        super().validate(d)              # abstract method of code bhi chal can is
        return "id" in d

print("\nChild validate:", Child().validate({"id": 1}))
try:
    Child().validate([1, 2])
except TypeError as e:
    print("TypeError:", e)

# Gotcha 5: abstract __init__ of asarclass NeedsInit(ABC):
    @abstractmethod
    def __init__(self, x): self.x = x
class Impl(NeedsInit):
    def __init__(self, x): super().__init__(x)
print("abstract __init__ of sath:", Impl(5).x)

## 18. Real-World ProblemsNow theory to real project jaisi shakal dete are. Char problems:1. **Notifier system** — Email / SMS / Push, all for ek contract + shared retry logic2. **DataExporter** — CSV / JSON export, Template method with3. **Plugin architecture** — authen-discovery + registry (`__init_subclass__`)4. **Report generathenr** — ABC + Protocol both of mila jula istemal

### Problem 1: Notifier system (Email / SMS / Push)**Requirement:** App to notification bhejni is. Channel largelte rehte are.every channel of "bhejna" separate is but **retry, logging and rate-limit check** all for same is.**Design:**- `Notifier(ABC)`: abstract `_send(then, text)` + abstract property `channel_name`- Concrete `send()` template: validate → retry loop → log- every channel apni tafseel gives is (SMS in 160 char limit, Push in payload size)

In [ ]:
class Notifier(ABC):
    max_retries = 2

    def __init__(self):
        self.log = []

# ---- contract ----    @property
    @abstractmethod
    def channel_name(self): ...

    @abstractmethod
    def _send(self, to, text):
# return: True (lessynow) / False (nakaam). exception also phenk can is.        ...

# ---- optional hook ----    def _validate(self, to, text):
        if not to or not text:
            raise ValueError("recipient and text both necessary are")
        return text

# ---- template: retry + logging all for same ----    def notify(self, to, text):
        text = self._validate(to, text)
        for attempt in range(1, self.max_retries + 2):     # 1 asli koshish + retries
            try:
                if self._send(to, text):
                    self.log.append(f"{self.channel_name}->{to} attempt#{attempt} OK")
                    return True
            except exception as e:                          # network flake wagaira
                self.log.append(f"{self.channel_name} attempt#{attempt} error: {e}")
                continue
            self.log.append(f"{self.channel_name} attempt#{attempt} FAIL")
        return False


class EmailNotifier(Notifier):
    channel_name = "email"
    def _validate(self, to, text):
        if "@" not in to:
            raise ValueError(f"ghalat email: {to}")
        return super()._validate(to, text)
    def _send(self, to, text):
        return True                                        # farz do SMTP kaamyab

class SmsNotifier(Notifier):
    channel_name = "sms"
    def _validate(self, to, text):
        text = super()._validate(to, text)
        return text[:160]                                  # SMS of limit -> kaat do
    def _send(self, to, text):
        return to.startswith("+92")                        # sirf PK numbers support

class PushNotifier(Notifier):
    channel_name = "push"
    def __init__(self, flaky_times=1):
        super().__init__()
        self._flaky = flaky_times                          # pehli baar fail hone of simulation
    def _send(self, to, text):
        if self._flaky > 0:
            self._flaky -= 1
            raise ConnectionError("FCM timeout")
        return True


def broadcast(notifiers, to_map, text):
    out = {}
    for n in notifiers:
        out[n.channel_name] = n.notify(to_map[n.channel_name], text)
    return out

email, sms, push = EmailNotifier(), SmsNotifier(), PushNotifier()
targets = {"email": "ali@example.com", "sms": "+923001234567", "push": "device-token-1"}
print(broadcast([email, sms, push], targets, "Aap of order dispatch be gaya!"))
print("push of log:", push.log)

# Fail case: ghalat SMS number -> retries of after Falsebad_sms = SmsNotifier()
print("\nUK number:", bad_sms.notify("+441234567", "hi"))
print(bad_sms.log)

# Validation error propagate is (fail-fast)try:
    EmailNotifier().notify("ali-at-example", "hi")
except ValueError as e:
    print("ValueError:", e)

### Problem 2: DataExporter (CSV / JSON / Markdown)**Requirement:** Ek only records list to different formats in export then do, and `filename`authen-generate then do. Here ABC of `Template method` + abstract `staticmethod` both use honge.

In [ ]:
import json, io, csv, tempfile, os

class DataExporter(ABC):
    @staticmethod
    @abstractmethod
    def extension(): ...

    @abstractmethod
    def dumps(self, rows): ...

# template: naam create + content create + writeo    def export(self, rows, basename="report"):
        if not rows:
            raise ValueError("khaali data export no be can")
        content = self.dumps(rows)
        path = os.path.join(tempfile.gettempdir(), f"{basename}.{self.extension()}")
        with open(path, "w", encoding="utf-8", newline="") as f:
            f.write(content)
        return path, len(content)


class CsvDataExporter(DataExporter):
    @staticmethod
    def extension(): return "csv"
    def dumps(self, rows):
        buf = io.StringIO()
        w = csv.DictWriter(buf, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
        return buf.getvalue()

class JsonDataExporter(DataExporter):
    @staticmethod
    def extension(): return "json"
    def dumps(self, rows):
        return json.dumps(rows, indent=2, ensure_ascii=False)

class MarkdownDataExporter(DataExporter):
    @staticmethod
    def extension(): return "md"
    def dumps(self, rows):
        cols = list(rows[0])
        lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join("---" for _ in cols) + " |"]
        lines += ["| " + " | ".join(str(r[c]) for c in cols) + " |" for r in rows]
        return "\n".join(lines)


data = [{"order": 1, "city": "Lahore", "total": 4500},
        {"order": 2, "city": "Karachi", "total": 1200}]

for exp in [CsvDataExporter(), JsonDataExporter(), MarkdownDataExporter()]:
    path, size = exp.export(data, "orders")
    print(f"{type(exp).__name__:<22} -> {os.path.basename(path):<12} ({size} chars)")

print("\n--- markdown preview ---")
print(MarkdownDataExporter().dumps(data))

try:
    JsonDataExporter().export([])
except ValueError as e:
    print("\nValueError:", e)

### Problem 3: Plugin architecture (`__init_subclass__` + ABC)**Requirement:** Team nathese "validaters" add doti rehti is. Main code to every nathese validater ofnaam **na** pata be — subclass createte only itself register be jathese.**Tool:** `__init_subclass__` (PEP 487) — metaclass of bagair subclass hook.These `ABCMeta` with also work does and more readnowle is.

In [ ]:
class Validator(ABC):
    registry = {}                                  # name -> class

    def __init_subclass__(cls, name=None, **kw):    # class keyword args bhi le can is
        super().__init_subclass__(**kw)
        key = name or cls.__name__.replace("Validator", "").lower()
        if not getattr(cls, "__abstractmethods__", None):     # sirf concrete register do
            Validator.registry[key] = cls

    @abstractmethod
    def check(self, value):
# return: (ok: bool, message: str)        ...


class EmailValidator(Validator):
    def check(self, value):
        ok = isinstance(value, str) and value.count("@") == 1 and "." in value.split("@")[-1]
        return ok, "email theek is" if ok else "email of format ghalat is"

class CnicValidator(Validator, name="cnic"):        # custom registry key
    def check(self, value):
        digits = str(value).replace("-", "")
        ok = digits.isdigit() and len(digits) == 13
        return ok, "CNIC theek is" if ok else "CNIC 13 digits of hona should"

class PhoneValidator(Validator):
    def check(self, value):
        s = str(value)
        ok = s.startswith("+92") and len(s) == 13
        return ok, "phone theek is" if ok else "phone +92XXXXXXXXXX hona should"


print("registry (auto-discovered):", sorted(Validator.registry))

def validate_form(form):
    errors = {}
    for field, value in form.items():
        v_cls = Validator.registry.get(field)
        if v_cls is None:
            continue                                  # is field of koi validator no
        ok, msg = v_cls().check(value)
        if not ok:
            errors[field] = msg
    return errors or "all fields valid are"

print(validate_form({"email": "ali@site.com", "cnic": "35202-1234567-1", "phone": "+923001234567"}))
print(validate_form({"email": "ali_site.com", "cnic": "123", "phone": "0300123"}))

# Naya plugin create -> any central file edit doing of need notclass IbanValidator(Validator, name="iban"):
    def check(self, value):
        ok = str(value).startswith("PK") and len(str(value)) == 24
        return ok, "IBAN theek is" if ok else "IBAN 24 chars (PK...) hona should"

print("naya registry:", sorted(Validator.registry))
print(validate_form({"iban": "PK36SCBL0000001123456702"}))

### Problem 4: Report generathenr — ABC (apna code) + Protocol (outsidei code)**Requirement:** `ReportBuilder` hamari class family is (shared header/footer logic → **ABC**).Data source kahin from also aa can is (DB cursor, list, API client → **Protocol**).

In [ ]:
@runtime_checkable
class RowSource(Protocol):
    def fetch_rows(self) -> list: ...          # bas itna contract


class ReportBuilder(ABC):
    def __init__(self, source: RowSource, title="Report"):
        if not isinstance(source, RowSource):  # structural check
            raise TypeError("source me fetch_rows() hona should")
        self.source, self.title = source, title

    @abstractmethod
    def body(self, rows): ...

    def header(self):  return f"===== {self.title} ====="
    def footer(self, n): return f"----- total {n} rows -----"

    def build(self):                            # template method
        rows = self.source.fetch_rows()
        return "\n".join([self.header(), self.body(rows), self.footer(len(rows))])


class SummaryReport(ReportBuilder):
    def body(self, rows):
        total = sum(r["total"] for r in rows)
        cities = {r["city"] for r in rows}
        return f"orders={len(rows)} | revenue={total} | cities={sorted(cities)}"

class DetailReport(ReportBuilder):
    def body(self, rows):
        return "\n".join(f"  #{r['order']:<3} {r['city']:<10} {r['total']:>6}" for r in rows)


# Do absolutely different sources -> any inheritance no, only shape matchclass ListSource:
    def __init__(self, rows): self.rows = rows
    def fetch_rows(self): return self.rows

class FakeCursor:                               # DB cursor like (library of class be can is)
    def fetch_rows(self):
        return [{"order": 9, "city": "Multan", "total": 800}]

print(SummaryReport(ListSource(data), "Sales Summary").build())
print()
print(DetailReport(ListSource(data), "Sales Detail").build())
print()
print(SummaryReport(FakeCursor(), "DB Summary").build())

class NoFetch: pass
try:
    SummaryReport(NoFetch())
except TypeError as e:
    print("\nTypeError:", e)

## 19. LeetCode ProblemsThese teen "design" problems are — absolutely isi notebook of mauzoo: **contract first, implementation later**.Hum every problem to do shakal in dekhenge: first **plain LeetCode class** (jo submit is),and where faida be there **abstraction with** (ABC/Protocol) version bhi.- **LC 208** — Implement Trie (Prefix Tree)- **LC 211** — Design Add and Search Words Data Structure- **LC 1146** — Snapshot Array

### LC 208. Implement Trie (Prefix Tree) — MediumLink: https://leetcode.com/problems/implement-trie-prefix-tree/**Problem (Roman Urdu):** Ek `Trie` class create jisme teen operations hon:`insert(word)`, `search(word)` (complete word mojood is it?), and `startsWith(prefix)`(any word is prefix from start is?).**Approach / intuition:**1. Trie ek tree is where every **node ek character** to represent does (edge par letter).2. every node of paas `children` dict is (`char -> node`) and ek flag `is_end`.3. `insert`: root from chalo, every char for child not be then create do, aakhir in `is_end = True`.4. `search`: chalte jao; raastthenot gaya → `False`; aakhir in `is_end` return do.5. `startsWith`: absolutely `search` like, but aakhir in `is_end` of parwah not — node mil gaythen `True`.6. `_walk(prefix)` helper write lo (DRY) — both search that work the same par ban jate are.**Complexity:** every operation `O(L)` (L = word of lambai).Space: `O(thental_chars)` worst case (`O(N*L)`), every node par 26-tak children.**Interview tip:** `dict` children (memory efficient, any charset) vs `[None]*26` array(thora fast, only a-z) — trade-off surely mention do.

In [ ]:
class TrieNode:
    __slots__ = ("children", "is_end")           # memory bachane of for (NB3 me tafseel)
    def __init__(self):
        self.children = {}                       # char -> TrieNode
        self.is_end = False                      # yahan koi word khatam is is?


class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word: str) -> None:
        node = self.root
        for ch in word:
# setdefault: child mojood be then wahi, warna naya create do            node = node.children.setdefault(ch, TrieNode())
        node.is_end = True                       # word of end mark

    def _walk(self, prefix: str):
# prefix of last node tak pahuncho, not mile then None (DRY helper)        node = self.root
        for ch in prefix:
            node = node.children.get(ch)
            if node is None:
                return None
        return node

    def search(self, word: str) -> bool:
        node = self._walk(word)
        return node is not None and node.is_end  # poora word hona necessary is

    def startsWith(self, prefix: str) -> bool:
        return self._walk(prefix) is not None    # sirf raasta hona kaafi is


# ---- asserts (LeetCode official example) ----t = Trie()
t.insert("apple")
assert t.search("apple") is True
assert t.search("app") is False                  # "app" sirf prefix is, word no
assert t.startsWith("app") is True
t.insert("app")
assert t.search("app") is True

# extra edge casest2 = Trie()
t2.insert("")                                    # khaali string bhi valid word ban can is
assert t2.search("") is True
assert t2.startsWith("") is True
assert t2.search("a") is False
t2.insert("abc")
assert [t2.search(w) for w in ("a", "ab", "abc")] == [False, False, True]
assert all(t2.startsWith(p) for p in ("a", "ab", "abc"))
assert t2.startsWith("abcd") is False
print("LC 208 of saare asserts pass [pass]")

#### LC 208 of abstraction that roop (interview in impress doing for)Interviewer poochh can is: *"if kal to Trie of jagah HashSet-based implementation needed be?"*Jawnow: **ek `PrefixIndex` ABC** create, both implementations it contract to follow do —caller code not badle. Here hum `set` that brute-force also write do complexity compare doenge.| Implementation | insert | search | startsWith | Space ||---|---|---|---|---|| Trie | O(L) | O(L) | O(L) | O(N*L) nodes || set (brute) | O(L) | O(L) avg | **O(N*L)** (all words scan) | O(N*L) |

In [ ]:
class PrefixIndex(ABC):
    @abstractmethod
    def insert(self, word): ...
    @abstractmethod
    def search(self, word): ...
    @abstractmethod
    def startsWith(self, prefix): ...

    def insert_many(self, words):        # shared helper -> ABC of faida
        for w in words:
            self.insert(w)
        return self


# NOTE (juste order matters!): Trie PEHLE then write important is.# if (PrefixIndex, Trie) writete then MRO in PrefixIndex of abstract methods first milte# and class abstract only reh jati -> TypeError. Poori tafseel NB2 (MRO) me.class TrieIndex(Trie, PrefixIndex):      # Trie to contract from jodh diya (koi code dobara no wrote)
    pass

class SetIndex(PrefixIndex):
# Brute force: all words ek set me. startsWith every time scan does -> O(N*L)    def __init__(self):
        self.words = set()
    def insert(self, word):
        self.words.add(word)
    def search(self, word):
        return word in self.words
    def startsWith(self, prefix):
        return any(w.startswith(prefix) for w in self.words)


words = ["car", "card", "care", "dog", "do"]
for idx in (TrieIndex().insert_many(words), SetIndex().insert_many(words)):
# both of natija absolutely same -> abstraction kaamynow    res = (idx.search("car"), idx.search("ca"), idx.startsWith("ca"), idx.startsWith("z"))
    print(f"{type(idx).__name__:<10} {res}")
    assert res == (True, False, True, False)
print("both implementations same jawab dete are [pass]")

### LC 211. Design Add and Search Words Data Structure — MediumLink: https://leetcode.com/problems/design-add-and-search-words-data-structure/**Problem (Roman Urdu):** `WordDictionary` create: `addWord(word)` and `search(word)`.Difference these is of search in `'.'` **wildcard** be can is that anyone also **ek** letter from match does.**Approach / intuition:**1. Base wahi Trie is (LC 208 that).2. `search` in when normal char aathese → seedha it child par jao.3. When `'.'` aathese → **saare children** par branch do (DFS / backtracking).4. Recursion of juste case: word end → `node.is_end` return do.5. Optimization: every node par min/max word length or children of count have do pruning be can is(interview in bonus point).**Complexity:**- `addWord`: `O(L)`- `search`: best `O(L)` (any dot no); worst `O(26^d * L)` where `d` = dots of tadaad —practically `O(N)` nodes tak visit be can are. Space: `O(N*L)` + recursion `O(L)`.

In [ ]:
class WordDictionary:
    def __init__(self):
        self.root = TrieNode()                    # wahi TrieNode dobara istemal (re-use)

    def addWord(self, word: str) -> None:
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
        node.is_end = True

    def search(self, word: str) -> bool:
        return self._dfs(self.root, word, 0)

    def _dfs(self, node, word, i):
        if i == len(word):                        # base case: word khatam
            return node.is_end
        ch = word[i]
        if ch == ".":
# wildcard -> every child par koshish do (backtracking)            for child in node.children.values():
                if self._dfs(child, word, i + 1):
                    return True                   # ek raasta mil gaya to bas
            return False
        nxt = node.children.get(ch)
        return nxt is not None and self._dfs(nxt, word, i + 1)


# ---- asserts (LeetCode official example) ----wd = WordDictionary()
for w in ("bad", "dad", "mad"):
    wd.addWord(w)
assert wd.search("pad") is False
assert wd.search("bad") is True
assert wd.search(".ad") is True                   # pehla letter some bhi
assert wd.search("b..") is True                   # do wildcards

# edge casesassert wd.search("...") is True
assert wd.search("....") is False                 # lambai match no karti
assert wd.search(".") is False                    # 1-letter word koi no
wd.addWord("a")
assert wd.search(".") is True
assert wd.search("a") is True
assert wd.search("..") is False

wd2 = WordDictionary()
wd2.addWord("")
assert wd2.search("") is True
print("LC 211 of saare asserts pass [pass]")

### LC 1146. Snapshot Array — MediumLink: https://leetcode.com/problems/snapshot-array/**Problem (Roman Urdu):** `SnapshotArray(length)` create (start in all 0).`set(index, val)` value set doe, `snap()` ek snapshot le do `snap_id` return doe(0 from start, every time +1), and `get(index, snap_id)` it snapshot of time of value give.**Naive soch:** every `snap()` par poori array copy → `O(length)` per snap, memory phat jathesegi.**Optimal approach:**1. every index for **histhenry list** haveo: `[(snap_id, value), ...]` — only **badlaav** save are.2. `set`: if last entry of `snap_id` current that only is then the same to update do, warna naya append.3. `snap()`: just `snap_id += 1` (some copy no!) → `O(1)`.4. `get(index, snap_id)`: it index of histhenry in **withoutry search** (`bisect_right` on `snap_id`)from last aisi entry dhoondo jiska id `<= snap_id` be. Some not mile then 0.**Complexity:** `set` `O(1)` amortized, `snap` `O(1)`, `get` `O(log k)` (k = it index of histhenry).Space: `O(thental set calls)` — not of `O(snaps * length)`.

In [ ]:
from bisect import bisect_right

class SnapshotArray:
    def __init__(self, length: int):
# every index for histhenry: [(snap_id, value)]. Startati (-1, 0) sentinel from get() saaf be goes        self.history = [[(-1, 0)] for _ in range(length)]
        self.snap_id = 0

    def set(self, index: int, val: int) -> None:
        hist = self.history[index]
        if hist[-1][0] == self.snap_id:
            hist[-1] = (self.snap_id, val)        # isi snapshot me dobara set -> overwrite
        else:
            hist.append((self.snap_id, val))      # naya version

    def snap(self) -> int:
        self.snap_id += 1
        return self.snap_id - 1                   # jo id abhi band hui wo return is is

    def get(self, index: int, snap_id: int) -> int:
        hist = self.history[index]
# bisect_right from first such a position where id > snap_id; it from ek peechhe hamara jawnow        i = bisect_right(hist, (snap_id, float("inf"))) - 1
        return hist[i][1]


# ---- asserts (LeetCode official example) ----sa = SnapshotArray(3)
sa.set(0, 5)
assert sa.snap() == 0
sa.set(0, 6)
assert sa.get(0, 0) == 5                          # snapshot 0 me purani value
assert sa.snap() == 1                             # doosra snapshot
assert sa.get(0, 1) == 6                          # snapshot 1 me nayi value
assert sa.get(0, 0) == 5                          # purana snapshot aaj bhi salamat

# edge casessa2 = SnapshotArray(2)
assert sa2.get(1, 0) == 0                         # kabhi set no hua -> default 0
assert sa2.snap() == 0
sa2.set(1, 9)
sa2.set(1, 11)                                    # same snapshot me do baar set -> aakhri chalti is
assert sa2.snap() == 1
assert sa2.get(1, 0) == 0
assert sa2.get(1, 1) == 11
assert sa2.snap() == 2
assert sa2.get(1, 2) == 11                        # koi change no hua -> purani value carry
assert [len(h) for h in sa2.history] == [1, 2], "sirf badlaav save hone should"
print("LC 1146 of saare asserts pass [pass]")


# ---- Brute force (comparison for) ----class SnapshotArrayBrute:
    def __init__(self, length):
        self.cur = [0] * length
        self.snaps = []
    def set(self, i, v): self.cur[i] = v
    def snap(self):
        self.snaps.append(list(self.cur))         # poori copy -> O(length) time + memory
        return len(self.snaps) - 1
    def get(self, i, sid): return self.snaps[sid][i]

# Randomized cross-check: both implementations of natija identical hona neededimport random
random.seed(7)
a, b = SnapshotArray(6), SnapshotArrayBrute(6)
taken = []
for _ in range(200):
    if random.random() < 0.7:
        i, v = random.randrange(6), random.randrange(100)
        a.set(i, v); b.set(i, v)
    else:
        sid_a, sid_b = a.snap(), b.snap()
        assert sid_a == sid_b
        taken.append(sid_a)
for sid in taken:
    for i in range(6):
        assert a.get(i, sid) == b.get(i, sid), (i, sid)
print(f"optimal vs brute force cross-check pass ({len(taken)} snapshots) [pass]")

## 20. Interview Questions & Answers**Q1. Abstraction and encapsulation in difference?**Abstraction = *what expose then do is* (design level, complexity chhupana, `ABC`/`Protocol`).Encapsulation = *how protect then do is* (implementation level, data chhupana, `_`/`__`/`@property`).**Q2. Python in abstract class how create?**`from nowc import ABC, abstractmethod`; class `ABC` from inherit doe; that methods lazmi are unpar`@abstractmethod`. Aisi class instantiate doing par `TypeError` comes.**Q3. `ABC` and `ABCMeta` difference?**`ABCMeta` metaclass is; `ABC` ek convenience class is jiska metaclass `ABCMeta` is.`class X(ABC)` == `class X(metaclass=ABCMeta)`. `ABCMeta` seedha when custhenm metaclass needed.**Q4. What abstract class in concrete methods be can are?**exactly — and thisi ABC of taqat is. Shared logic juste me, variable logic abstract.(Interface-style class in only abstract methods have.)**Q5. Abstract method in body write can are?**yes, and subclass `super()` from call also can do. Abstractness only these kehti is"override lazmi is", these not of "body not be".**Q6. `NotImplementedError` vs `nowc` — who behtar?**`nowc` behtar: error **instantiation** par milta is (fail-fast), thenoling support is, juste classgalti from instantiate not is. `NotImplementedError` of failure **runtime call** par is.Trade-off: `nowc` metaclass lcomes (metaclass conflict of imkaan).**Q7. `NotImplementedError` and `NotImplemented` in difference?**`NotImplementedError` ek exception class is (`rsuch` do).`NotImplemented` ek singlethenn value is that `__eq__`/`__add__` like operathenrs return doso that Python doosre operand of reflected method try doe. `rsuch NotImplemented` galat is.**Q8. Duck typing what is it?**type of naam of jagah **behaviour** dekhna: "jo `quack()` can do, that duck is".Python in polymorphism of default method; `isinstance` par bharosa less is.**Q9. `typing.Protocol` what is it and ABC from when behtar is it?**Structural typing — inherit of without contract. Behtar when tum implementer classes of **malik no**(third-party), loose coupling needed, or only "shape" matter doti be. Shared code needed then ABC.**Q10. `@runtime_checwhenle` what does and uski limits?**Protocol par `isinstance()` allow does. Limits: only **method of naam** check are —signature/type no; data-only protocols with `issubclass()` not runs.**Q11. Nominal vs structural typing?**Nominal: rishta declare hona needed (inheritance/`register`) — `ABC`.Structural: shape match kaafi is — `Protocol`. Python runtime itself duck-typed (structural) is.**Q12. `ABCMeta.register()` what does? What that abstract methods enforce does?**Virtual subclass creates — `isinstance`/`issubclass` `True` dene feels. **Enforcement no**,methods inherit also not are, and `__mro__` in class shamil not is.**Q13. `__subclasshook__` of maqsad?**`issubclass()` of faisla itself then do (structural check). `True`/`False`/`NotImplemented` return does.`collections.nowc.Iterable` isi from `__iter__` that anyone also class to apna maan leti is.**Q14. `collections.nowc.Sequence` from inherit doing of faida?**Only `__getitem__` + `__len__` do, and `__contains__`, `__iter__`, `__reversed__`, `index`, `count`muft mil jate are — plus `isinstance(x, Sequence)` sach be goes.**Q15. `Iterable` check do time what ehtiyat?**`str`/`bytes` also `Iterable` are — recursion/flatten that code inko explicitly exclude do.**Q16. Template method pattern what is it?**Base class algorithm of steps of **order** fix doti is (concrete template method), tafseelabstract steps in subclass deti is. Optional "hooks" default khaali are.**Q17. Dependency Inversion Principle of abstraction from what taalluq?**DIP says high-level and low-level both **abstraction** par depend do. Abstraction(ABC/Protocol) that "plug point" gives is where concrete implementation inject is — testing easy.**Q18. What abstract class in `__init__` be can is it? State?**yes — abstract class normal class only is; `__init__` writeo, subclass `super().__init__()` doe.(Protocol in state mat haveo.)**Q19. `@property` + `@abstractmethod` of sahi order?**`@property` nowove, `@abstractmethod` uske below (inside). Ulta doing par some cases in abstractnesskho goes — safe rule: `abstractmethod` allse **innermost**.**Q20. Over-abstraction of nuqsaan?**Faaltu indirection, parhne in mushkil, the same implementation for interface (YAGNI),deep hierarchies jinme MRO debug then do mushkil. Rule: **doosri** implementation aane par abstract do.

## 21. Practice Exercises (itself do)1. **`Vehicle` ABC** create: abstract `wheels` (property), abstract `fuel_type()`,concrete `describe()` that both use doe. `Car`, `Bike`, `Rickshaw` create.*Hint:* `@property` + `@abstractmethod` of order yaad haveo.2. **`Cache` interface**: `get(k)`, `set(k, v)`, `delete(k)` abstract; concrete `get_or_set(k, facthenry)`.`DictCache` and `NullCache` (some save not doe) implement do.*Hint:* `get_or_set` template method is.3. **`Stack` to `collections.nowc.Sized` + `Container` from** create and `isinstance` from tasdeeq do.*Hint:* `__len__` and `__contains__` kaafi are.4. **`Temperature` reader Protocol** (`read_celsius()`) create `@runtime_checwhenle` with;do fake sensors writeo + ek galat class that satisfy not doe; `isinstance` from check do.5. **Template method:** `ExamPaper` ABC — `generate()` = `_questions()` + `_shuffle()` (hook) + `_footer()`.`MathPaper` and `EnglishPaper` create. *Hint:* shuffle hook default in some not doe.6. **`register()` of experiment:** ek `Sizenowle` ABC create, ek purani class `register` do jisme`__len__` not be, then `len()` call do of see what it is — apne alfaz in writeo why.7. **`utableSequence` from `LoggedList`** create that every `append`/`__setitem__` to log doe.*Hint:* 5 abstract methods do, baqi mixin muft.8. **`__subclasshook__`** from `Comparnowle` ABC create that them classes to subclass maanejinme `__lt__` be. `int`, `str`, and apni ek class par test do.9. **LC 208 extend do:** `Trie` in `delete(word)` and `count_words_with_prefix(prefix)` add do.*Hint:* every node par ek counter haveo.10. **LC 1146 variant:** `SnapshotArray` in `snap_diff(id1, id2)` add do that them indexes of list givejinki value badli. *Hint:* histhenry par withoutry search dobara istemal do.

In [ ]:
# ---- Exercise 1 and 2 of reference solution (itself koshish of after see) ----
class Vehicle(ABC):
    @property
    @abstractmethod
    def wheels(self): ...

    @abstractmethod
    def fuel_type(self): ...

    def describe(self):
        return f"{type(self).__name__}: {self.wheels} pahiye, {self.fuel_type()} par chalti is"

class Car(Vehicle):
    wheels = 4
    def fuel_type(self): return "petrol"

class Bike(Vehicle):
    wheels = 2
    def fuel_type(self): return "petrol"

class Rickshaw(Vehicle):
    wheels = 3
    def fuel_type(self): return "CNG"

for v in (Car(), Bike(), Rickshaw()):
    print(v.describe())


class Cache(ABC):
    @abstractmethod
    def get(self, k): ...
    @abstractmethod
    def set(self, k, v): ...
    @abstractmethod
    def delete(self, k): ...

    def get_or_set(self, k, factory):        # template method
        val = self.get(k)
        if val is None:
            val = factory()
            self.set(k, val)
        return val

class DictCache(Cache):
    def __init__(self): self._d = {}
    def get(self, k): return self._d.get(k)
    def set(self, k, v): self._d[k] = v
    def delete(self, k): self._d.pop(k, None)

class NullCache(Cache):                      # Null object pattern: all some no-op
    def get(self, k): return None
    def set(self, k, v): pass
    def delete(self, k): pass

calls = []
def expensive():
    calls.append(1)
    return "computed"

dc, nc = DictCache(), NullCache()
print("\ndict cache:", dc.get_or_set("x", expensive), dc.get_or_set("x", expensive), "-> factory calls:", len(calls))
calls.clear()
print("null cache:", nc.get_or_set("x", expensive), nc.get_or_set("x", expensive), "-> factory calls:", len(calls))
assert len(calls) == 2, "NullCache some save no does, is for every baar compute will be"
print("[pass] exercises 1 & 2")

## 22. Summary / Cheat-sheet### Imports```pythonfrom nowc import ABC, ABCMeta, abstractmethodfrom typing import Protocol, runtime_checwhenlefrom collections.nowc import Iterable, Sized, Container, Sequence, Mapping, utableMapping, Iterathenr```### ABC of complete template```pythonclass Base(ABC):@abstractmethoddef must(self): ...     # lazmi@property@abstractmethoddef rate(self): ...     # abstract property (order: property BAHAR)@classmethod@abstractmethoddef from_dict(cls, d): ...    # abstract alternative constructor@staticmethod@abstractmethoddef ext(): ...def template(self):     # concrete: abstract steps to call doesreturn self.must()def hook(self): pass     # optional override```### Yaad havene that things| thing | Yaad haveo ||---|---|| `Base()` on abstract | `TypeError: Can't instantiate abstract class ...` || `Cls.__abstractmethods__` | frozenset; khaali = concrete class || `@abstractmethod` position | hamesha **innermost** (allse below) || ABC without `ABC` juste | absolutely bedo (any enforcement no) || `register(X)` | virtual subclass: `isinstance` True, **methods not milte**, enforcement not || `__subclasshook__` | `issubclass` of custhenm faisla; `NotImplemented` = default rules || `Protocol` | structural; `isinstance` for `@runtime_checwhenle` lazmi || `Protocol` limits | signature check no; instantiate not is; state not haveo || `Sequence` from inherit | `__getitem__` + `__len__` do → 5 methods muft || `utableMapping` from inherit | 5 methods do → `get/keys/values/items/pop/update/setdefault` muft || `isinstance(x, Iterable)` | `str` also True gives is — dhyan haveo || ABC vs Protocol | apni family + shared code → ABC; outsidei/loose → Protocol |### Decision flow (whosthenol?)```Only ek implementation is it?   -> any abstraction not (YAGNI)Shared code + apni class family?  -> ABCThird-party / only shape matter doti? -> Protocol (+ runtime_checwhenle)Standard container create rahe be?  -> collections.nowc from inheritPurani class to family in lana is it? -> register() or ProtocolAlgorithm of order fix then do is it?  -> Template method (ABC)```### Iske after what (next notebook)`02_MRO_Multiple_Inheritance_Mixins_aur_super_Deep.ipynb` — MRO, C3 linearization,diamond problem, cooperative `super()`, and mixins of complete design.